In [20]:
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

/ebio/abt6/lvaness/anaconda3/envs/jupyter/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.1
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [21]:
def augustus_refs(df):
    out_df= pd.DataFrame()
    accessions=df["accession"].unique().tolist()
    for acc in accessions:    
        busco = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/augustus_mappings/busco_{acc}.txt", header=None)
        house = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/augustus_mappings/housekeeping_{acc}.txt", header=None)
        nlr = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/augustus_mappings/nlrs_{acc}.txt", header=None)

        busco["type"] = "busco"
        house["type"] = "housekeeping"
        nlr["type"] = "nlr"
        gene_type = pd.concat([busco,house,nlr])
        gene_type.columns = ["ref_isoform","type"]
        gene_type = gene_type.groupby("ref_isoform")["type"].apply(', '.join).reset_index()

        gene_type["ref_gene"] = gene_type["ref_isoform"].str.split(".").str[0]
        gene_type.loc[gene_type["type"] == "busco, nlr"] = "nlr"
        gene_type.loc[gene_type["type"] == "housekeeping, nlr"] = "nlr"
        gene_type.loc[gene_type["type"] == "busco, housekeeping"] = "housekeeping"

        sub=df.loc[df["accession"]==acc]
        sub = pd.merge(left=sub, right=gene_type, how='left', left_on='associated_gene', right_on='ref_gene')
        
        out_df=pd.concat([out_df,sub])
    return out_df

In [22]:
def liftoff_refs(df):
    # get_df
    accessions=df["accession"].unique().tolist()

    busco = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/busco_Araport11_20221221.txt", header=None)
    house = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/arat_housekeeping.txt", header=None)
    nlr = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/nlrs_in_orthogroups_20221221.txt", header=None)

    busco["type"] = "busco"
    house["type"] = "housekeeping"
    nlr["type"] = "nlr"
    gene_type = pd.concat([busco,house,nlr])
    gene_type.columns = ["ref_isoform","type"]
    gene_type = gene_type.groupby("ref_isoform")["type"].apply(', '.join).reset_index()

    gene_type["ref_gene"] = gene_type["ref_isoform"].str.split(".").str[0]
    gene_type.loc[gene_type["type"] == "busco, nlr"] = "nlr"
    gene_type.loc[gene_type["type"] == "housekeeping, nlr"] = "nlr"
    gene_type.loc[gene_type["type"] == "busco, housekeeping"] = "housekeeping"
        
    gene_ref = pd.merge(left=df, right=gene_type, how='left', left_on='associated_gene', right_on='ref_gene')
    
    return gene_ref

In [23]:
def subsample(df, len_nlr, refname):
        ### Random subsampling form the genome
        # Preparation
        n=len_nlr
        exons_all, length_all, subs_all, cats_all=[],[],[],[]
        exons_dict, length_dict, subs_dict, cats_dict={},{},{},{}
        exons_final, length_final, subs_final, cats_final={},{},{},{}
        np.random.seed(42)
        seeds = np.random.choice(range(999999), size=10000, replace=False)

        # get snapshot of one iteration
        for i in range(len(seeds)):
            sub_df = df.sample(n, random_state=seeds[i], axis=0)
            exons_all.append(sub_df["exons"])
            length_all.append(sub_df["length"])
            subs_all.append(sub_df["subcategory"])
            cats_all.append(sub_df["structural_category"])

        # count occurences of all snapshots
        for subset in exons_all:
            for gene in subset:
                if gene in exons_dict.keys():
                    exons_dict[gene] +=1
                else:
                    exons_dict[gene] = 1

        for subset in length_all:
            for gene in subset:
                if gene in length_dict.keys():
                    length_dict[gene] +=1
                else:
                    length_dict[gene] = 1

        for subset in subs_all:
            for gene in subset:
                if gene in subs_dict.keys():
                    subs_dict[gene] +=1
                else:
                    subs_dict[gene] = 1
                    
        for subset in cats_all:
            for gene in subset:
                if gene in cats_dict.keys():
                    cats_dict[gene] +=1
                else:
                    cats_dict[gene] = 1

        # average the accumulated snaphot counts by the number of snapshots and get the probability by dividing all by all genes
        for key in exons_dict.keys():
            exons_final[key] = exons_dict[key]/len(seeds)

        for key in length_dict.keys():
            length_final[key] = length_dict[key]/len(seeds)

        for key in subs_dict.keys():
            subs_final[key] = subs_dict[key]/len(seeds)
            
        for key in cats_dict.keys():
            cats_final[key] = cats_dict[key]/len(seeds)
            
        exons_iters = pd.DataFrame.from_dict(exons_final, orient='index',columns=[f"{refname}_iters_exons"])
        exons_iters[f"{refname}_iters_exons_proportion"] = exons_iters[f"{refname}_iters_exons"]/n
        
        length_iters = pd.DataFrame.from_dict(length_final, orient='index',columns=[f"{refname}_iters_length"])
        length_iters[f"{refname}_iters_length_proportion"] = length_iters[f"{refname}_iters_length"]/n
        
        subs_iters = pd.DataFrame.from_dict(subs_final, orient='index',columns=[f"{refname}_iters_subs"])
        subs_iters[f"{refname}_iters_subs_proportion"] = subs_iters[f"{refname}_iters_subs"]/n
        
        cats_iters = pd.DataFrame.from_dict(cats_final, orient='index',columns=[f"{refname}_iters_cats"])
        cats_iters[f"{refname}_iters_cats_proportion"] = cats_iters[f"{refname}_iters_cats"]/n
        
        return exons_iters, length_iters, subs_iters, cats_iters

In [24]:
tama = pd.read_pickle("../sqanti/tama_augustus_sqanti_full.pkl")

In [25]:
pasa = pd.read_pickle("../pasa/pasa_augustus_sqanti_full.pkl")

In [34]:
names=["tama_augustus","pasa_augustus"]
runs=["../sqanti/tama_augustus_sqanti_full.pkl","../pasa/pasa_augustus_sqanti_full.pkl"]

out=pd.DataFrame()
for name, run in zip(names,runs):
    # load full df
    ref=pd.read_pickle(f"{run}")
    
    # add new column
    ref["cat_assign"]="None"
    # and fill new colum based on 4 categories
    ref.loc[(ref["structural_category"] != "antisense") & (ref["structural_category"] != "fusion") & (ref["structural_category"] != "intergenic"), "cat_assign"]="one2one"
    ref.loc[(ref["structural_category"] == "antisense"), "cat_assign"]="antisense"
    ref.loc[(ref["structural_category"] == "fusion"), "cat_assign"]="fusion"
    ref.loc[(ref["structural_category"] == "intergenic") | (ref["structural_category"] == "genic_intron"), "cat_assign"]="novel"
    
    # in some cases, transcripts were reported as one2one even though the reads are spanning two genes.
    # In these cases, we have "fusion reads", but splicing results in perfect cds for one gene.
    # filter them and add replace the assignment
    ref["association"] = ref["associated_gene"].str.split("_").str.len()
    ref.loc[(ref["cat_assign"]=="one2one")&(ref["association"]>1), "cat_assign"] = "fusion"
        
    ref =augustus_refs(ref)
        
    ref["type"].fillna('other', inplace=True)
    ref["run"]=name
    out=pd.concat([out,ref])

In [35]:
out

,gene,isoform,accession,associated_gene,associated_transcript,exons,length,isoform_read_support,gene_read_support,structural_category,subcategory,Sqanti_ORF_seq,cat_assign,association,ref_isoform,type,ref_gene,run
0,G1,G1.1,at6923,g1,g1.t1,6,1663,1.0,2.0,full-splice_match,alternative_3end5end,MEDQVGFGFRPNDEELVGHYLRNKIEGNTSRDVEVAISEVNICSYD...,one2one,1,NaN,other,NaN,tama_augustus
1,G1,G1.2,at6923,g1,g1.t1,6,1588,1.0,2.0,full-splice_match,alternative_3end5end,MEDQVGFGFRPNDEELVGHYLRNKIEGNTSRDVEVAISEVNICSYD...,one2one,1,NaN,other,NaN,tama_augustus
2,G10,G10.1,at6923,g9,g9.t1,3,1640,3.0,182.0,full-splice_match,alternative_3end5end,MATAFAPTKLTATVPLHGSQENRLLLPIRLAPPSSFLGSTRSLSLR...,one2one,1,NaN,other,NaN,tama_augustus
3,G10,G10.10,at6923,g9,g9.t1,2,1175,1.0,182.0,incomplete-splice_match,3prime_fragment,MCAQMYYRGKMFGFVHLYNGQEAVSTGFIKLLTKSDSVVSTYRDHV...,one2one,1,NaN,other,NaN,tama_augustus
4,G10,G10.11,at6923,g9,g9.t1,3,1473,2.0,182.0,full-splice_match,alternative_3end,MATAFAPTKLTATVPLHGSQENRLLLPIRLAPPSSFLGSTRSLSLR...,one2one,1,NaN,other,NaN,tama_augustus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45542,ATMG01350,ATMG01350.1,col0,ATMG01350,ATMG01350.1,1,438,NaN,NaN,full-splice_match,mono-exon,MTKREYNSQPEMKEEVLAYLLQLSASLVLPVAIWLIAAGQIFTCLR...,one2one,1,NaN,other,NaN,pasa_augustus
45543,ATMG01360,ATMG01360.1,col0,ATMG01360,ATMG01360.1,1,1584,NaN,NaN,full-splice_match,mono-exon,MKNLVRWLFSTNHKDIGTLYFIFGAIAGVMGTCFSVLIRMELARPG...,one2one,1,NaN,other,NaN,pasa_augustus
45544,ATMG01370,ATMG01370.1,col0,ATMG01370,ATMG01370.1,1,336,NaN,NaN,full-splice_match,mono-exon,NaN,one2one,1,NaN,other,NaN,pasa_augustus
45545,ATMG01400,ATMG01400.1,col0,ATMG01400,ATMG01400.1,1,318,NaN,NaN,full-splice_match,mono-exon,MKYHFSSMEPWWKREFSFCIPAIYIKMASISLFQNSWLKMKHLPSC...,one2one,1,NaN,other,NaN,pasa_augustus


In [39]:
out.loc[(out["run"]=="pasa_augustus")&(out["cat_assign"]=="one2one")&(out["subcategory"]=="intron_retention")]["type"].value_counts()

other           126171
busco            40218
housekeeping      2601
nlr               2078
Name: type, dtype: int64

## at9900 Tama

### general

In [41]:
out.loc[(out["run"]=="tama_augustus")&(out["cat_assign"]=="one2one")]["subcategory"].value_counts()

alternative_3end5end                1047470
at_least_one_novel_splicesite        720374
alternative_3end                     375197
mono-exon                            352849
3prime_fragment                      333676
intron_retention                     303474
multi-exon                            74906
combination_of_known_splicesites      42073
reference_match                       39060
5prime_fragment                       37427
alternative_5end                      37118
internal_fragment                      6079
mono-exon_by_intron_retention          2141
combination_of_known_junctions          567
Name: subcategory, dtype: int64

In [42]:
out.loc[(out["run"]=="pasa_augustus")&(out["cat_assign"]=="one2one")]["subcategory"].value_counts()

reference_match                     220035
intron_retention                    171068
at_least_one_novel_splicesite       137862
mono-exon                           135394
alternative_3end5end                133067
alternative_3end                     26729
multi-exon                           13729
combination_of_known_splicesites      6867
3prime_fragment                       3790
5prime_fragment                       3215
alternative_5end                      2414
combination_of_known_junctions         171
internal_fragment                       30
Name: subcategory, dtype: int64

In [44]:
t9900=out.loc[(out["run"]=="tama_augustus")&(out["cat_assign"]=="one2one")&(out["accession"] == "at9900")].copy()
# novel transcripts
novel_transcripts = t9900["associated_transcript"].value_counts()["novel"]
# ref_genes wit novel transcripts
ref_genes_w_novel_transcripts=t9900.loc[t9900["associated_transcript"]=="novel", "associated_gene"].value_counts().index.unique()
len(ref_genes_w_novel_transcripts)
# tama_genes wit novel transcripts
tama_genes_w_novel_transcripts=t9900.loc[t9900["associated_transcript"]=="novel", "gene"].value_counts().index.unique()
len(tama_genes_w_novel_transcripts)
# novel genes
t9900["gene_nov"]= t9900.associated_gene.str.split("_").str[0]
novel_genes=t9900.loc[t9900["gene_nov"]=="novelGene"]["gene"].value_counts().index.unique().tolist()
len(novel_genes)
t9900.loc[t9900["associated_transcript"]=="novel", "subcategory"].value_counts()
t9900.loc[t9900["associated_transcript"]!="novel", "subcategory"].value_counts()

alternative_3end5end    62764
alternative_3end        21943
3prime_fragment         18824
mono-exon               15076
5prime_fragment          2032
intron_retention         2015
alternative_5end          942
reference_match           422
internal_fragment         294
Name: subcategory, dtype: int64

In [45]:
### subset:
for i in ["all", "nlr", "busco", "housekeeping"]:
    print(f"####{i}####")
    if i == "all":
        df = t9900
    else:
        df = t9900[t9900["type"] == i].copy()
    
    # Tama genes
    tama_genes = df["gene"].value_counts().index.unique()
    print(f"Tama genes: {len(tama_genes)}")
    
    # augustus genes
    ref_genes = df["associated_gene"].value_counts().index.unique()
    print(f"Ref genes: {len(ref_genes)}")
    
    # transcripts
    print(f"Transcripts: {len(df)}")
    
    # novel transcripts
    novel_transcripts = len(df.loc[df["associated_transcript"]=="novel"])
    print(f"Novel transcripts: {novel_transcripts}")
    
    # ref_genes wit novel transcripts
    ref_genes_w_novel_transcripts=df.loc[df["associated_transcript"]=="novel", "associated_gene"].value_counts().index.unique()
    len(ref_genes_w_novel_transcripts)
    print(f"Reference genes with novel transcripts: {len(ref_genes_w_novel_transcripts)}")
    
    # tama_genes wit novel transcripts
    tama_genes_w_novel_transcripts=df.loc[df["associated_transcript"]=="novel", "gene"].value_counts().index.unique()
    len(tama_genes_w_novel_transcripts)
    print(f"Tama genes with novel transcripts: {len(tama_genes_w_novel_transcripts)}")
    
    # novel genes
    df["gene_nov"]= df.associated_gene.str.split("_").str[0]
    novel_genes=df.loc[df["gene_nov"]=="novelGene"]["gene"].value_counts().index.unique().tolist()
    len(novel_genes)
    print(f"Novel genes: {len(novel_genes)}")
    
    print("Novel transcripts subategories:")
    print(df.loc[df["associated_transcript"]=="novel", "subcategory"].value_counts())
    
    print("Non novel transcripts subategories:")
    print(df.loc[df["associated_transcript"]!="novel", "subcategory"].value_counts())
    print("\n")

####all####
Tama genes: 15934
Ref genes: 16176
Transcripts: 192703
Novel transcripts: 68391
Reference genes with novel transcripts: 10286
Tama genes with novel transcripts: 10203
Novel genes: 0
Novel transcripts subategories:
at_least_one_novel_splicesite       41507
intron_retention                    14503
mono-exon                            5297
multi-exon                           4374
combination_of_known_splicesites     2660
mono-exon_by_intron_retention          50
Name: subcategory, dtype: int64
Non novel transcripts subategories:
alternative_3end5end    62764
alternative_3end        21943
3prime_fragment         18824
mono-exon               15076
5prime_fragment          2032
intron_retention         2015
alternative_5end          942
reference_match           422
internal_fragment         294
Name: subcategory, dtype: int64


####nlr####
Tama genes: 123
Ref genes: 122
Transcripts: 1290
Novel transcripts: 573
Reference genes with novel transcripts: 79
Tama genes with novel t

### at9900 Pasa

In [46]:
pasa

,gene,isoform,accession,associated_gene,associated_transcript,exons,length,structural_category,subcategory,Sqanti_ORF_seq
0,g1,g1.t1,at6923,g1,g1.t1,6,1663,full-splice_match,alternative_3end5end,MEDQVGFGFRPNDEELVGHYLRNKIEGNTSRDVEVAISEVNICSYD...
1,g10,g10.t1,at6923,g10,novel,4,603,novel_not_in_catalog,at_least_one_novel_splicesite,MSTVGELACSYAVMILEDEGIAITADKIATLVKAAGVSIESYWPML...
2,g100,g100.t1,at6923,g100,g100.t1,1,1787,full-splice_match,mono-exon,MGSLVKAYYCNQMHHNSIIPHFHHYRPSSLQNPISLFAITSPSSSE...
3,g1000,g1000.t1,at6923,g1000,novel,5,1653,novel_not_in_catalog,at_least_one_novel_splicesite,MLSGNNGNTAPPVFMNENQLQYQTNLRSNQLHLLGNMGGGCTVDPV...
4,g1001,g1001.t1,at6923,g1001,novel,4,788,novel_not_in_catalog,at_least_one_novel_splicesite,NaN
...,...,...,...,...,...,...,...,...,...,...
45167,ATMG01350,ATMG01350.1,col0,ATMG01350,ATMG01350.1,1,438,full-splice_match,mono-exon,MTKREYNSQPEMKEEVLAYLLQLSASLVLPVAIWLIAAGQIFTCLR...
45168,ATMG01360,ATMG01360.1,col0,ATMG01360,ATMG01360.1,1,1584,full-splice_match,mono-exon,MKNLVRWLFSTNHKDIGTLYFIFGAIAGVMGTCFSVLIRMELARPG...
45169,ATMG01370,ATMG01370.1,col0,ATMG01370,ATMG01370.1,1,336,full-splice_match,mono-exon,NaN
45170,ATMG01400,ATMG01400.1,col0,ATMG01400,ATMG01400.1,1,318,full-splice_match,mono-exon,MKYHFSSMEPWWKREFSFCIPAIYIKMASISLFQNSWLKMKHLPSC...


In [47]:
pasa_type = pd.merge(left=pasa, right=gene_type, how='left', left_on='associated_transcript', right_on='ref_isoform')

In [48]:
p9900=out.loc[(out["run"]=="pasa_augustus")&(out["cat_assign"]=="one2one")&(out["accession"] == "at9900")]

In [50]:
p9900

,gene,isoform,accession,associated_gene,associated_transcript,exons,length,isoform_read_support,gene_read_support,structural_category,subcategory,Sqanti_ORF_seq,cat_assign,association,ref_isoform,type,ref_gene,run
0,g1,g1.t1,at9900,g1,g1.t1,6,1685,NaN,NaN,full-splice_match,alternative_3end5end,MEDQVGFGFRPNDEELVGHYLRNKIEGNTSRDVEVAISEVNICSYD...,one2one,1,NaN,other,NaN,pasa_augustus
1,g10,g10.t1,at9900,g10,novel,4,612,NaN,NaN,novel_not_in_catalog,at_least_one_novel_splicesite,MSTVGELACSYAVMILEDEGIAITADKIATLVKAAGVSIESYWPML...,one2one,1,NaN,other,NaN,pasa_augustus
2,g100,g100.t1,at9900,g100,g100.t1,1,1626,NaN,NaN,full-splice_match,mono-exon,MKLACFVFLILSSLVSSSLATAPPNASIYESFLQCFSNQTGAPPEK...,one2one,1,NaN,other,NaN,pasa_augustus
3,g1000,g1000.t1,at9900,g1000,g1000.t1,1,2398,NaN,NaN,full-splice_match,mono-exon,MIHTKTGSGRRILTFPTVEPSESISIVTLLDSLIQLAGDILTFKSK...,one2one,1,NaN,other,NaN,pasa_augustus
4,g1001,g1001.t1,at9900,g1001,novel,14,1942,NaN,NaN,novel_not_in_catalog,at_least_one_novel_splicesite,MKRQRAIELDRVKKTMLNIDWDDALGDEEVPELEIIATDKIPPREP...,one2one,1,NaN,other,NaN,pasa_augustus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47507,g31852,g31852.t1,at9900,g31852,g31852.t1,2,1203,NaN,NaN,full-splice_match,reference_match,MSTPAQSDLVKVFTDVASNYRGTSDIDLHAVYRDMRANSPVLQDNF...,one2one,1,NaN,other,NaN,pasa_augustus
47508,g31853,g31853.t1,at9900,g31853,g31853.t1,1,1308,NaN,NaN,full-splice_match,mono-exon,MKVEDLILVSVDDHAIEPPNAFARHMPARFKGREPHVIKKGDRDVW...,one2one,1,NaN,other,NaN,pasa_augustus
47509,g31854,g31854.t1,at9900,g31854,g31854.t1,1,1230,NaN,NaN,full-splice_match,mono-exon,MAAVERLPFCGRFVRSDCACVERERAHMAISQFSYSSIPPEAEALR...,one2one,1,NaN,other,NaN,pasa_augustus
47510,g31855,g31855.t1,at9900,g31855,g31855.t1,2,1767,NaN,NaN,full-splice_match,reference_match,MGCEGGEWPAELWAALDESGFTQAGVTEALGGAEASWADMYVLVRA...,one2one,1,g31855.t1,busco,g31855,pasa_augustus


In [51]:
### subset:
for i in ["all", "nlr", "busco", "housekeeping"]:
    print(f"####{i}####")
    if i == "all":
        df_a = p9900
    else:
        df_a = p9900[p9900["type"] == i].copy()
    
    df = df_a.loc[df_a["subcategory"] != "reference_match"].copy()
    #df = df_a
    
    # Pasa genes
    pasa_genes = df["gene"].value_counts().index.unique()
    print(f"Pasa genes: {len(pasa_genes)}")
    
    # augustus genes
    ref_genes = df["associated_gene"].value_counts().index.unique()
    print(f"Ref genes: {len(ref_genes)}")
    
    # transcripts
    print(f"Transcripts: {len(df)}")
    
    # novel transcripts
    novel_transcripts = len(df.loc[df["associated_transcript"]=="novel"])
    print(f"Novel transcripts: {novel_transcripts}")
    
    # ref_genes wit novel transcripts
    ref_genes_w_novel_transcripts=df.loc[df["associated_transcript"]=="novel", "associated_gene"].value_counts().index.unique()
    len(ref_genes_w_novel_transcripts)
    print(f"Reference genes with novel transcripts: {len(ref_genes_w_novel_transcripts)}")
    
    # pasa_genes wit novel transcripts
    pasa_genes_w_novel_transcripts=df.loc[df["associated_transcript"]=="novel", "gene"].value_counts().index.unique()
    len(pasa_genes_w_novel_transcripts)
    print(f"Pasa genes with novel transcripts: {len(pasa_genes_w_novel_transcripts)}")
    
    # novel genes
    df["gene_nov"]= df.associated_gene.str.split("_").str[0]
    novel_genes=df.loc[df["gene_nov"]=="novelGene"]["gene"].value_counts().index.unique().tolist()
    len(novel_genes)
    print(f"Novel genes: {len(novel_genes)}")
    
    print("Novel transcripts subategories:")
    print(df.loc[df["associated_transcript"]=="novel", "subcategory"].value_counts())
    
    print("Non novel transcripts subategories:")
    print(df.loc[df["associated_transcript"]!="novel", "subcategory"].value_counts())
    print("\n")

####all####
Pasa genes: 20814
Ref genes: 20814
Transcripts: 36506
Novel transcripts: 17601
Reference genes with novel transcripts: 8061
Pasa genes with novel transcripts: 8061
Novel genes: 0
Novel transcripts subategories:
intron_retention                    8364
at_least_one_novel_splicesite       8015
multi-exon                           814
combination_of_known_splicesites     408
Name: subcategory, dtype: int64
Non novel transcripts subategories:
alternative_3end5end    8056
mono-exon               7611
alternative_3end        1457
intron_retention        1313
3prime_fragment          243
5prime_fragment          189
alternative_5end          34
internal_fragment          2
Name: subcategory, dtype: int64


####nlr####
Pasa genes: 118
Ref genes: 118
Transcripts: 237
Novel transcripts: 135
Reference genes with novel transcripts: 55
Pasa genes with novel transcripts: 55
Novel genes: 0
Novel transcripts subategories:
intron_retention                    69
at_least_one_novel_splicesite